# core

> the vault: one SQLite file holding everything you have read, and the retrieval over it

The user wants to understand what the `vishalakshi` project does. let me go through and get the whole information



In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A `Vault` **is** a [litesearch](https://github.com/Karthik777/litesearch) `Index` — docs → nodes →
chunks, FTS5 and a usearch HNSW index over one SQLite file — with three things added: a `kind` on
every document, sibling shelves in the same file, and an entity graph over it.

Nothing here re-implements retrieval. `add`, `search`, `sections`, `read`, `toc` and `context` are
`Index`'s, and every default they carry — 512-character chunks, `pre()` on the keyword leg, an ANN
vector leg, a document tree, float16 — is the configuration litesearch's `evals/` measured as best
or tied-best across three genres. What the vault adds is the facet filter pushed into each of
them, and the acquisition, extraction and code verbs the other modules patch on.

In [ ]:
#| export
import json, re, time, uuid, warnings
from collections import Counter
import numpy as np
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from litesearch import (Index, DTYPE, dir2files, hash_embed, static_embedder, build_graph,
        resolve_entities, topic_nodes, FastEncode, DOC_EXTS, embedding_gemma)

In [ ]:
#| export
KINDS = ('web', 'pdf', 'arxiv', 'youtube', 'file', 'code', 'data', 'note')
_window, DFLT_ENC = re.compile(r'^Pages \d+(?:–\d+)?:'), 'minishlab/potion-multilingual-128M'

def tidy_bc(bc:str) -> str:
    "Drop `build_tree`'s `Pages n–m:` window placeholders from a breadcrumb: real nodes, noise in a citation."
    return ' › '.join(dict.fromkeys(p for p in map(str.strip, (bc or '').split('›')) if p and not _window.match(p)))

def kinds(kind) -> L:
    "A kind filter as a list — `'note'`, `'note,web'` and `['note','web']` all work."
    return L(kind.split(',') if isinstance(kind, str) else kind).filter()

# The encoder is not the lever. Across four encoders litesearch measures a spread of 0.018–0.046
# weighted MRR, and the static model *wins* one genre outright while indexing ~1,700x cheaper. So
# there is one default, and this is a table of overrides for an experiment — not a menu to be
# picked from per corpus. `DFLT_ENC` is what `static_embedder()` loads, so a vault and a bare
# `Index` over the same file agree without being told to.
ENCODERS = {
    # alias          what litesearch loads what it reads better than the default
    'default':       DFLT_ENC,
    'multilingual':  DFLT_ENC,                              # 100+ languages, Devanagari included
    'retrieval':     'minishlab/potion-retrieval-32M',      # tuned for search, not for similarity
    'science':       'minishlab/potion-science-32M',        # papers: abstracts, methods, results
    'code':          'minishlab/potion-code-16M-v2',        # identifiers; what kosha embeds with
    'gemma':         embedding_gemma,                       # ONNX, ~300M: the most faithful, the slowest
}

_MODEL_MAP = {m['model']: m for m in ENCODERS.values() if isinstance(m, dict)}

def _is_enc(o) -> bool: return isinstance(o, AttrDict) and 'model' in o and 'dims' in o
def enc_spec(model=None) -> tuple:
    '`(what to load, how)` for an encoder: an `ENCODERS` alias, a model2vec id, a litesearch model dict, or an embedder you built yourself.'
    if model and not isinstance(model, (str, bytes, dict)) and hasattr(model, 'encode'): return model, 'ready'
    spec = ENCODERS.get(model, model if model is not None else DFLT_ENC)
    spec = _MODEL_MAP.get(spec, spec) if isinstance(spec, str) else spec
    return spec, ('onnx' if isinstance(spec, dict) else 'static')

In [ ]:
#| export
class HashEmbed:
    "litesearch's `hash_embed` behind an `.encode`, so an offline vault is an encoder like any other."
    def __init__(self, dims:int=256, dtype=DTYPE): self.dims, self.dtype = dims, dtype
    def encode(self, xs, **kw): return hash_embed(list(xs), ndim=self.dims, dtype=self.dtype)

def mk_encoder(model=None,          # an ENCODERS alias, a model2vec id, a litesearch model dict, or an embedder
               dims:int=256,        # dims for the hashing fallback only
               offline:bool=False,  # skip the download attempt entirely
               dtype=DTYPE,         # stored width; litesearch's default everywhere
) -> AttrDict:
    '''The best encoder available as `AttrDict(model, dims, method, name, note)`.

    Only `model` is the encoder. Turning it into the document and query functions is `Index`'s
    job, which is why nothing here builds them: one thing that knows how to embed, not two.'''
    if offline:
        return AttrDict(model=HashEmbed(dims, dtype), dims=dims, method='hash', name='hash',
        note=f'char-n-gram hashing ({dims}d) — lexical only; pass encoder= or restore network access for real semantics')
    spec, how = enc_spec(model)
    nm = spec['model'] if isinstance(spec, dict) else (spec if isinstance(spec, str) else type(spec).__name__)
    try:
        m = spec if how == 'ready' else FastEncode(spec, dtype=dtype) if how == 'onnx' else static_embedder(spec)
        v, meth = m.encode(['probe']), 'onnx' if isinstance(m, FastEncode) else 'model2vec'
        return AttrDict(model=m, dims=int(v.shape[-1]), method=meth, name=nm,
                        note=f'{nm} ({v.shape[-1]}d, {np.dtype(dtype)}, {meth})')
    except Exception as e:
        warnings.warn(f'could not load {nm} ({type(e).__name__}: {str(e)[:100]}); falling back to hash_embed')
        return mk_encoder(dims=dims, offline=True, dtype=dtype)

In [ ]:
#| hide
# mk_encoder returns the encoder; `Index` is what turns it into doc and query functions. So the
# offline fallback has to *be* an encoder — something with `.encode` — not a pair of functions.
_h = mk_encoder(offline=True)
test_eq(_h.model.encode(['a', 'b']).shape, (2, 256))
test_eq(_h.model.encode(['a']).dtype, np.float16)
test_eq((_h.method, _h.dims, _h.name), ('hash', 256, 'hash'))

class _Fwd:                       # a model2vec-shaped embedder: `doc_encoder` drops kw for these
    def encode(self, xs): return np.zeros((len(xs), 8), dtype=np.float16)
test_eq((mk_encoder(_Fwd()).dims, mk_encoder(_Fwd()).method), (8, 'model2vec'))

In [ ]:
#| export
class Vault(Index):
    '''Everything you have read, in one SQLite file, searchable as one corpus.

    A litesearch `Index` with a `kind` on every document, shelves beside it in the same file, an
    entity graph over it, and the acquisition and extraction verbs patched on by the other modules.
    Nothing here re-implements retrieval: `add`, `search`, `sections`, `read`, `toc` and `context`
    are `Index`'s, with a facet filter pushed into them.'''
    def __init__(self,
                 path:str=None,       # vault file; None -> ~/.vishalakshi/vault.db
                 encoder=None,        # an ENCODERS alias, a model id, an mk_encoder() result, or None
                 store:str='store',   # chunk store name
                 offline:bool=False,  # never attempt a model download
                 dims:int=256,        # dims for the hashing fallback
                 db=None):            # an open litesearch Database to share; shelves pass the vault's
        self.enc = encoder if _is_enc(encoder) else mk_encoder(encoder, dims=dims, offline=offline)
        super().__init__(ifnone(path, Path.home()/'.vishalakshi'/'vault.db'),
                         encoder=self.enc.model, name=store, db=db)
        self._register()

    def _where(self, kind) -> str| None:
        'A chunk-store `WHERE` for a kind filter — pushed into the search, not applied after it.'
        return None if not kinds(kind) else f'doc_id IN (SELECT id FROM {self.t.prefix}docs WHERE {_kw(kind)})'

    def __repr__(self):
        s = self.stats()
        return (f"Vault({self.path!r}: {s['docs']} docs, {s['chunks']} chunks, {s['entities']} entities, encoder={self.enc.method})")

def _kw(kind) -> str: return 'kind IN (%s)' % ','.join(map(repr, kinds(kind)))

In [ ]:
#| export
@patch
def add(self:Vault,
        src,                  # text, `[(page_no, text)]`, a file path, or a directory
        title:str=None,       # document title; defaults to the filename, or the first line of text
        source:str=None,      # url or path; defaults to the title. Identity is hashed over it
        kind:str=None,        # one of KINDS — the facet you filter and report on
        meta:dict=None,       # provenance: the query that found it, when, which tier fetched it
        force:bool=False,     # re-ingest a source already present
        **kw                  # forwarded to litesearch add_doc (chunker, summarize, with_heading)
) -> dict:
    '''Ingest anything into the vault: tree, chunks, embeddings, ANN index.

    `src` decides the route, the way `Index.add` does — a directory goes to `add_dir`, a file to
    `add_file`, and anything else is treated as the text of one document. Identity is
    content-addressed over `source|title`, so re-adding the same page is a no-op rather than a
    duplicate, and `force=True` re-ingests a source already present.'''
    # a document's text is not a path, and asking the filesystem about a 40kB "filename" raises
    p = Path(src) if isinstance(src, (str, Path)) and len(str(src)) < 255 and '\n' not in str(src) else None
    if p is not None and p.is_dir():  return self.add_dir(str(p), kind=kind, **kw)
    if p is not None and p.is_file(): return self.add_file(str(p), title=title, kind=kind, **kw)
    ttl = title or _first_line(src)
    return self.db.add_doc(src, ttl, source=source, kind=kind or 'file', store=self.name,
                           emb_fn=self.emb, meta=meta, force=force, **kw)

def _first_line(src, n:int=80) -> str:
    'A title for text that came without one — the first non-empty line, as `toc()` will show it.'
    txt = src if isinstance(src, str) else '\n'.join(t for _, t in (src or []))
    return next((l.strip().lstrip('# ') for l in txt.splitlines() if l.strip()), 'untitled')[:n]

@patch
def assets(self:Vault, name:str=None) -> Path:
    'Where extracted assets (PDF images, downloaded papers) go: beside the vault file. litesearch picks it.'
    return self.db.assets(name)

@patch
def add_file(self:Vault, path:str, title:str=None, kind:str=None, **kw) -> dict:
    'Ingest one local file into the vault: tree, chunks, embeddings, ANN index.'
    return self.db.add_file(path, title=title, kind=kind, store=self.name, emb_fn=self.emb, **kw)

@patch
def add_files(self:Vault,
              files,                # paths to ingest, all onto *this* shelf
              kind:str=None,        # override the kind inferred from the extension
              n_workers:int=None,   # parse workers; 0 is serial, None picks by parse-heavy file count
              embed_batch:int=2000, # chunks embedded and written per flush; 0 writes per document
              **kw                  # forwarded to add_file
) -> L:
    '''Ingest a list of files onto this shelf, batched the way litesearch batches a whole tree.

    A file at a time costs an embedder call, a transaction and an ANN index rebuild *per document*.
    `Database.add_dir` already does the batching — a parse pool, chunks deferred and embedded
    `embed_batch` at a time across documents, one index rebuild at the end — and it takes `files=`
    instead of insisting on walking a tree. So the batching is its, and only the routing, which is
    per file and cannot be, stays out here.'''
    files = L(files).map(Path)
    # an empty list would still reach `rebuild_index`, which is a real cost for no documents
    if not files: return L()
    return L(self.db.add_dir(files=files, store=self.name, kind=kind, emb_fn=self.emb,
                             n_workers=n_workers, embed_batch=embed_batch, **kw))

@patch
def add_dir(self:Vault, dir:str, types:str=DOC_EXTS, kind:str=None,
            route:bool=True,      # send Sanskrit sources to the Sanskrit shelf, file by file
            **kw) -> L:
    """Ingest every document under a directory. `dir2files` skips dotfiles, tests, build and dist.

    Routing is per *file*, not per directory, because a directory is not a kind: a corpus checkout
    mixes GRETIL editions with the README that describes them, and filing the whole tree by its
    first document would put one or the other on a shelf whose encoder cannot read it. `grab` can
    only route what it can see from the outside — a path — so the second look happens here.

    Each shelf is then batched on its own, which is what it has to be: the encoder, the chunk store
    and the ANN index are all per shelf."""
    fs = dir2files(dir, types=types)
    if not route or self.name != 'store': return self.add_files(fs, kind=kind, **kw)
    sa, rest = L(), L()
    for f in fs: (sa if is_sanskrit_file(f) else rest).append(f)
    out = self.add_files(rest, kind=kind, **kw)
    if sa: out += self.route('sanskrit').add_files(sa, kind=kind, **kw)
    return out

@patch
def note(self:Vault,
         text:str,            # what you want to remember
         title:str=None,      # defaults to the first line
         tags:list=None,      # free-form tags, kept in the doc's meta
) -> dict:
    'Write a note into the vault so it is searched alongside the corpus.'
    ttl = title or (text.strip().splitlines() or ['note'])[0].lstrip('# ')[:80]
    return self.add(text.strip(), ttl, source=f'note:{uuid.uuid4().hex[:12]}', kind='note', meta=dict(tags=list(tags or [])))

`context` is the retrieval an LLM should be handed: whole sections plus what they connect to.
Sections carry `text`, `breadcrumb`, `pages` and `filename` alongside their tree neighbourhood, and
`related` holds sections reached by embedding similarity.

`related` is vector-reached, not graph-reached. litesearch's `context` defaults `graph=False` and
the vault does not override it, because the graph leg costs 0.070 to 0.160 weighted MRR on
ordinary known-item queries — negative in every genre and flavour measured, monotonically worse as
its weight rises. It wins only on *bridge* queries, where the answer shares no word with the
question. `connect()` is still worth running: it is what `map()` and entity browsing read, and it
is reachable by name through `db.graph_search` when you know your traffic looks like that.

`code=None` appends federated code sections *when kosha has already indexed the repo*, after the
prose ones, so the `[n]` numbering an answer cites is unaffected by whether that leg ran.
`rerank=True` reorders the chunk hits before they are rolled up into sections — worth +0.026 to
+0.077 weighted MRR at roughly 10x the query latency.

In [ ]:
#| export
@patch
def search(self:Vault,
           q:str,              # query
           limit:int=10,       # hits to return
           kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
           chars:int=300,      # chars of each hit kept as `snippet`
           rerank:bool=False,  # reorder the candidates with a cross-encoder (see below)
           **kw                # forwarded to litesearch doc_search
) -> L:
    '''Chunk-level hybrid search (FTS5 + vectors, RRF-fused), each hit carrying its breadcrumb.

    A hit is the handle, not the text: `node_id` reads the section, `doc_id` the document. The
    fused score is the only one worth keeping — the legs' own `rank` and `_dist` are on different
    scales and only one leg sets each. Same shape as `related`, so a result set reads the same
    whether you reached it by query or by proximity.

    This overrides `Index.search`, which searches the flat chunk store. A vault always wants the
    tree: span merging, and a breadcrumb that makes a hit citable. `rerank=True` is worth +0.026
    to +0.077 weighted MRR at roughly 10x the latency.'''
    hits = self.db.doc_search(q, self.qemb(q), limit=limit, store=self.name, dtype=DTYPE,
                              where=self._where(kind), rerank=rerank, **kw)
    return L(AttrDict(node_id=h.get('node_id'), doc_id=h.get('doc_id'), page=h.get('page'),
                      breadcrumb=tidy_bc(h.get('breadcrumb')), score=h.get('_rrf_score'),
                      snippet=(h.get('content') or '')[:chars]) for h in hits)

@patch
def sections(self:Vault, q:str, limit:int=5, kind:str=None, per:int=3, rerank:bool=False, **kw) -> list:
    'Ranked *sections* rather than chunks — the unit worth reading, each with a `read` handle.'
    secs = self.db.sections(q, self.qemb(q), limit=limit, per=per, store=self.name, dtype=DTYPE,
                            where=self._where(kind), rerank=rerank, **kw)
    for s in secs: s['breadcrumb'] = tidy_bc(s.get('breadcrumb'))
    return secs

@patch
def context(self:Vault,
            q:str,              # the question
            sections:int=6,     # operative sections returned
            related:int=8,      # related sections reached by graph + vector
            kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
            max_read:int=6000,  # chars of assembled text per section
            code:int=None,      # code sections to append; None -> 4 if kosha has indexed the repo
            shelves:int=2,      # sections to append from each *other* shelf; 0 -> none
            dir:str=None,       # repo for the code legs; None -> the cwd repo
            rerank:bool=False,  # reorder the chunk hits before they are rolled up into sections
            **kw                # forwarded to litesearch context
) -> AttrDict:
    'The retrieval an LLM should be handed: whole sections plus what they connect to. sections carry `text, breadcrumb, pages, filename` and their tree neighbourhood;'
    keep = None if not kinds(kind) else {r['id'] for r in self.t.docs(where=_kw(kind), select='id')}
    ctx = self.db.context(q, self.qemb(q), store=self.name, related=related, max_read=max_read,
                          sections=sections*3 if keep else sections, rerank=rerank, **kw)
    if keep is not None:
        ctx.results = ctx.results.filter(lambda r: r.doc_id in keep)[:sections]
        ctx.related = ctx.related.filter(lambda r: r.doc_id in keep)[:related]
    for r in (*ctx.results, *ctx.related): r.breadcrumb = tidy_bc(r.breadcrumb)
    ctx.encoder, ctx.code, ctx.shelves = self.enc.note, 0, 0
    if shelves:
        found = self.elsewhere(q, limit=shelves)
        ctx.results, ctx.shelves = ctx.results + found, len(found)
    if code or code is None:
        from vishalakshi.code import code_sections, kosha_indexed
        if kosha_indexed(dir):
            hits = code_sections(self, q, n=code or 4, dir=dir)
            ctx.results, ctx.code = ctx.results + hits, len(hits)
    return ctx

@patch
def related(self:Vault, node_id:str, limit:int=8, clip=300) -> L:
    'Sections nearest an existing one .Reuses the vectors usearch already holds, so nothing is re-embedded.'
    out = {}
    for r in self.store(where=f'node_id={node_id!r}', select='rowid as rowid'):
        for n in self.store.ann_neighbors(r['rowid'], limit=limit*3, dtype=DTYPE, columns=['content', 'node_id', 'doc_id']):
            nid = n.get('node_id')
            if nid and nid != node_id and nid not in out:
                out[nid] = dict(node_id=nid, doc_id=n.get('doc_id'), dist=n.get('_dist'),
                                breadcrumb=tidy_bc(self.db.breadcrumb(nid, self.name)), snippet=(n.get('content') or '')[:clip])
            if len(out) >= limit: return L(out.values())
    return L(out.values())

@patch
def read(self:Vault, node_id:str, max_chars:int=6000, store:str=None) -> dict:
    'Assemble a whole section back out of its chunks.`store` opens a section on another shelf, so a `node_id` from `elsewhere()` can be read without opening that shelf.'
    return self.db.read(node_id, store=store or self.name, max_chars=max_chars)


A `node_id` names a section and a `doc_id` names a document, so retrieval hands back the first and
acquisition the second. These three close the gap: one document row, one whole document, and a place
to record what you later concluded about it.

`doc` takes three kinds of reference because three different things hand you one: retrieval returns
`doc_id`s, acquisition returns the url or path it filed, and you remember the title. Exact matches
are tried first, so a title that happens to contain another's is still reachable.

`read` returns a section; `document` returns all of them, in document order. Headings go back in
because a chunk is stored bare and a heading is often the only thing that says what a number
*means* — an invoice reassembled without its `Total` line is a column of unlabelled figures — but
only the headings the document actually wrote, never `build_tree`'s window placeholders.
`disk=True` reads a file the vault does not hold, so it can be handed to a model without ingesting
it first, and `origin` says which of the two answered.

`set_meta` is where what a document *is* lands, rather than in `kind`: `kind` is the fixed facet
retrieval filters on, chosen by whatever acquired the document, while a document type is a
judgement — one a later, better model may revise.

In [ ]:
#| export
@patch
def doc(self:Vault, ref:str) -> dict:
    'One document row, by `doc_id`, exact `source`, or a title substring; `meta` already decoded.'
    q = str(ref or '').replace("'", "''")
    for w in (f"id='{q}'", f"source='{q}'", f"title LIKE '%{q}%'"):
        if r:=first(self.t.docs(where=w, order_by='added_at desc')): return dict(r, meta=json.loads(r['meta'] or '{}'))
    return None

@patch
def document(self:Vault,
             ref:str,               # doc_id, source (url or path), or a title substring
             max_chars:int=40000,   # cap on the text returned
             headings:bool=True,    # put the node titles back as markdown headings
             disk:bool=True,        # fall back to a path on disk the vault has never seen
) -> AttrDict:
    'One whole document, reassembled in document order — the unit a model reads to extract from.'
    d = self.doc(ref)
    if d is None:
        p = Path(ref or '')
        if not (disk and p.is_file()): raise ValueError(f'no document in the vault matching {ref!r}')
        txt = p.read_text(errors='replace')
        return AttrDict(doc_id=None, title=p.name, source=str(p), kind='file', meta={}, pages=None, origin='disk', nodes=0,
                        chars=len(txt), truncated=len(txt) > max_chars, text=txt[:max_chars])
    did = d['id'].replace("'", "''")
    chunks = {}
    for c in self.store(where=f"doc_id='{did}'", select='content, node_id, page, rowid as rowid'):
        chunks.setdefault(c['node_id'], []).append(c)
    nodes, parts = sorted(self.t.nodes(where=f"doc_id='{did}'"), key=lambda r: r['seq']), []
    for nd in nodes:
        if headings and nd['level'] and (t := (nd['title'] or '').strip()) and not _window.match(t):
            parts.append('#'*min(nd['level'], 6) + ' ' + t)
        parts += [c['content'] for c in sorted(chunks.get(nd['id'], []), key=lambda c: (c['page'] or 0, c['rowid']))]
    txt = '\n\n'.join(p for p in parts if (p or '').strip())
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], kind=d['kind'], meta=d['meta'],
                    pages=d['pages'], origin='vault', nodes=len(nodes), chars=len(txt), truncated=len(txt) > max_chars, text=txt[:max_chars])

@patch
def set_meta(self:Vault, doc_id:str, **kv) -> dict:
    "Merge key/values into one document's `meta`, and return the merged dict."
    did = (doc_id or '').replace("'", "''")
    r = first(self.t.docs(where=f"id='{did}'"))
    if not r: raise ValueError(f'no document {doc_id!r} in the vault')
    m = {**json.loads(r['meta'] or '{}'), **kv}
    self.t.docs.update(dict(id=doc_id, meta=json.dumps(m, default=str)))
    return m

One vault file, several shelves. A shelf is a *partition*: its own chunk store, its own tree and
its own ANN index, so a Sanskrit corpus and a folder of invoices do not dilute each other's
ranking, and `find`ing on one never returns the other.

It is deliberately not a second vector space. Every shelf is written by the same encoder, because
across four encoders litesearch measures a spread of only 0.018–0.046 weighted MRR and the static
default *wins* one genre outright at ~1,700x cheaper indexing. Where a corpus really does need
more, the answer measured on this vault's own Sanskrit shelf was not a bigger encoder but better
*text*: glosses in the index beat a 300M ONNX transformer without them.

`shelf(name)` reopens a shelf with whatever encoder wrote it, and warns if you hand it another —
distances across two vector spaces are meaningless, and the warning is the only sign. Since a
shelf's encoder cannot be migrated in place, `drop_shelf(name)` is how you change one: drop, then
re-ingest. `elsewhere` carries `store` on every row, because that is what `read(node_id, store=…)`
needs to open a citation that lives on another shelf.

In [ ]:
#| export
@patch
def _stores(self:Vault):
    'The registry of stores in this vault file and which encoder wrote each; created on first use.'
    t = self.db.t.vault_stores
    t.create(store=str, encoder=str, dims=int, method=str, added_at=float, pk='store', if_not_exists=True)
    return t

@patch
def _register(self:Vault):
    'Record which encoder wrote this store, and say so loudly when it is reopened with another.'
    try:
        t, now = self._stores(), time.time()
        r = first(t(where=f'store={self.name!r}'))
        if r and (r['encoder'], r['dims']) != (self.enc.name, self.enc.dims):
            warnings.warn(f"store {self.name!r} was written by {r['encoder']} ({r['dims']}d) but this Vault is "
                f"using {self.enc.name} ({self.enc.dims}d). Distances across the two are meaningless. "
                f"Re-ingest, or keep them apart with shelf('{self.name}-{self.enc.method}').")
        elif not r: t.insert(dict(store=self.name, encoder=self.enc.name, dims=self.enc.dims, method=self.enc.method, added_at=now), replace=True)
    except Exception: pass   # a read-only vault must still open; the registry is a convenience

@patch
def shelf(self:Vault, name:str, encoder:str=None, **kw) -> Vault:
    '''A sibling vault in the same file: its own store, its own tree, its own ANN index.

    A shelf is a *partition*, not a second encoder. It is reopened with whatever wrote it, and with
    the default otherwise — `encoder=` is for an experiment you are running deliberately, and it
    will warn if it disagrees with what is already on disk.'''
    was = first(self._stores()(where=f'store={name!r}')) or {}
    enc = encoder or was.get('encoder')
    if enc == 'hash': enc, kw = None, dict(kw, offline=True)   # nothing to load; do not try
    return Vault(self.path, encoder=enc, store=name, db=self.db, **kw)

@patch
def drop_shelf(self:Vault, name:str, force:bool=False) -> dict:
    '''Delete a shelf outright: its chunks, nodes, docs, entity graph, ANN index and registry row.

    The way to change a shelf's encoder is to drop it and re-ingest. An ANN index holds exactly one
    vector space, so writing 256d vectors into a shelf built at 512d does not migrate it — it makes
    every distance across the two meaningless, which is what `_register` warns about.'''
    if name == 'store' and not force: raise ValueError("refusing to drop the main shelf; pass force=True")
    pre = '' if name == 'store' else f'{name}_'
    have = {r['name'] for r in self.db.q("select name from sqlite_master where type='table'")}
    gone = []
    # the FTS virtual tables first: dropping one takes its shadow tables with it, and dropping the
    # content table out from under it first leaves them orphaned
    for tn in (f'{name}_fts', f'{pre}entities_fts', name, f'{pre}nodes', f'{pre}docs',
               f'{pre}entities', f'{pre}mentions', f'{pre}edges'):
        if tn in have:
            self.db.q(f'DROP TABLE IF EXISTS [{tn}]'); gone.append(tn)
    for r in self.db.q('select path from usearch_indices where name=?', [name]):
        try: Path(r['path']).unlink(missing_ok=True)
        except Exception: pass
    self.db.q('delete from usearch_indices where name=?', [name])
    try: self._stores().delete_where(f'store={name!r}')
    except Exception: pass
    return dict(shelf=name, dropped=gone)

@patch
def shelves(self:Vault) -> L:
    'Every store in this vault file, with the encoder that wrote it and how many documents it holds.'
    def n(s):
        p = '' if s == 'store' else f'{s}_'
        try: return self.db.t[f'{p}docs'].count
        except Exception: return 0
    return L(self._stores()(order_by='added_at')).map(lambda r: dict(r, docs=n(r['store'])))

# Shelf names, not encoder assignments. One encoder writes every shelf, because the measurement
# says the encoder is not the lever — a shelf earns its keep as a *partition*, so that a Sanskrit
# corpus and a folder of invoices do not dilute each other's ranking, not as a second vector space.
SHELVES = ('store',      # the main shelf: notes, pages, anything unrouted
           'papers',     # arXiv and papers
           'sanskrit',   # veda, commentary, translation — Devanagari and IAST alike
           'code',       # source filed as prose; kosha is the real code index, reached by federate
           'data')       # API harvests and record dumps


Where acquisition routes, and it is deliberately one entry. A route is only safe where the
destination is strictly better and nothing reads the old shelf expecting to still find it: `find`
and `sections` are single-shelf, so a write that quietly moves is a read that quietly comes back
empty. A PDF is as likely an invoice as a paper anyway — `DOCTYPE_SHELF` and `reshelf` are the
route for what only becomes clear once a document has been read.

In [ ]:
#|export
KIND_SHELF = {'arxiv': 'papers', 'sanskrit': 'sanskrit'}

def is_sanskrit_file(path) -> bool:
    'Whether litesearch has a Sanskrit reader for this file.'
    from litesearch.data import profile_for
    p = Path(path)
    return p.is_file() and (pr := profile_for(p)) is not None and (pr.kind or '') == 'sanskrit'

_facets_on = False
def sanskrit_facets() -> bool:
    '''Re-register litesearch's Sanskrit profiles *with* lemmas and Monier-Williams glosses.

    litesearch calls `register_profiles()` at import with no lemmatiser and no lexicon, so a verse
    is indexed with its metre and nothing else. Putting the English behind the Sanskrit into
    `metadata` is the single largest measured gain on this corpus — larger than changing the
    encoder: a static encoder *with* glosses beats a 300M ONNX transformer without them. The gloss
    is indexed for FTS and never embedded, so it costs no schema change and no vector width.

    Called once, the first time a file is routed to the Sanskrit shelf, because the data behind it
    is an ~83 MB download that nobody who is not reading Sanskrit should pay for. A failure leaves
    the metre-only profiles in place rather than raising: worse retrieval, not a failed ingest.'''
    global _facets_on
    if _facets_on: return True
    try:
        from litesearch.sanskrit import register_profiles, vidyut_pipe, mw_lexicon
        register_profiles(nlp=vidyut_pipe(), mw=mw_lexicon())
        _facets_on = True
    except Exception as e:
        warnings.warn(f'Sanskrit lemmas and glosses unavailable ({type(e).__name__}: {str(e)[:80]}); '
                      'indexing metre only. Retrieval by English paraphrase will be weaker.')
    return _facets_on

@patch
def route(self:Vault, kind:str) -> Vault:
    'The shelf a `kind` belongs on: `self`, unless `KIND_SHELF` sends it elsewhere.'
    nm = KIND_SHELF.get(kind)
    if nm == 'sanskrit': sanskrit_facets()   # before the ingest that is about to happen, not after
    return self.shelf(nm) if nm and self.name == 'store' and nm != self.name else self

@patch
def elsewhere(self:Vault,
              q:str,              # the question
              limit:int=2,        # sections taken from each other shelf
              shelves=True,       # True -> every other shelf; a list picks some
              max_read:int=2000,  # chars kept per section
) -> L:
    "Sections from the vault's *other* shelves, shaped like this one's."
    out, want = L(), (None if shelves is True else set(L(shelves)))
    for s in self.shelves():
        nm = s['store']
        if nm == self.name or not s['docs'] or (want is not None and nm not in want): continue
        for r in self.shelf(nm).sections(q, limit=limit): out.append(AttrDict(node_id=r['node_id'],
            doc_id=r['node_id'].split('#')[0], store=nm, title=r['title'], breadcrumb=f"{nm} › {r['breadcrumb']}",
            filename=None, pages=r['pages'], via=f'shelf:{nm}', text=' '.join(r['snippets'])[:max_read]))
    return out

In [ ]:
#| export
def _paged(tbl, batch:int=2000):
    'Rows out of `tbl` a page at a time, so nothing has to hold the whole corpus at once.'
    if not batch: yield from tbl(); return   # `batch=0` is "do not page", not "page by nothing"
    off = 0
    while (rows := list(tbl(limit=batch, offset=off, order_by='rowid'))):
        yield from rows
        off += len(rows)

@patch
def connect(self:Vault,
            resolve:bool=True,   # merge duplicate entities once the build is in
            topics:bool=True,    # (re)write the labelled topic nodes
            batch:int=2000,      # chunks per flush; keeps co-occurrence windows on disk, not in memory
            n_workers:int=None,  # extraction workers; 0 is serial, None picks by queue size
            **kw) -> dict:
    '''(Re)build the entity graph over everything in the vault.

    The corpus is streamed rather than materialised: `build_graph` reads `chunks` exactly once, and
    `batch` sends its windows to a scratch table, so a vault that has outgrown memory still finishes.
    Extraction is where the build spends its time, which is what `n_workers` splits — litesearch
    keeps it serial on its own when `terms_fn` is set, since an extractor rarely pickles.'''
    if not self.store.count: return dict(entities=0, mentions=0, edges=0, windows=0)
    if 'terms_fn' not in kw:
        try:
            from litesearch.sanskrit import is_sanskrit, sanskrit_terms
            head = ' '.join(c['content'] or '' for c in self.store(limit=20))
            if is_sanskrit(head): kw['terms_fn'] = sanskrit_terms()
        except Exception: pass          # vidyut is an extra; fall back to whatever litesearch defaults to
    self.db.get_graph(self.name, ndim=self.enc.dims, dtype=DTYPE)
    res = build_graph(self.db, _paged(self.store, batch), store=self.name, emb_fn=self.emb,
                      batch=batch, n_workers=n_workers, **kw)
    if resolve: res = dict(res, resolved=resolve_entities(self.db, store=self.name, dtype=DTYPE))
    if topics:
        g = self.db.get_graph(self.name)
        try: g.mentions.delete_where(f"entity_id IN (SELECT id FROM {g.prefix}entities WHERE kind='topic')")
        except Exception: pass
        try: g.entities.delete_where("kind='topic'")
        except Exception: pass
        res = dict(res, **topic_nodes(self.db, store=self.name, dtype=DTYPE))
    return res

def _map_from_graph(db, store, store_table, members:int=24) -> AttrDict|None:
    'Read topic clusters persisted by `connect()` — instant DB read, no re-clustering.'
    try: g = db.get_graph(store)
    except Exception: return None
    try: ents = L(g.entities(where="kind='topic'", order_by='freq desc'))
    except Exception: return None
    if not ents: return None
    eids = ','.join(repr(e['id']) for e in ents)
    cid_topic = {}
    for m in g.mentions(select='chunk_id, entity_id', where=f'entity_id IN ({eids})'):
        cid_topic.setdefault(m['entity_id'], []).append(m['chunk_id'])
    want = list({c for cs in cid_topic.values() for c in cs[:members]})
    rows = ({r['id']: r for r in store_table(select='id, content, doc_id',
             where=f"id IN ({','.join(repr(i) for i in want)})")} if want else {})
    clusters = L(AttrDict(centroid=None, size=e['freq'],
                          label=e['content'].removeprefix('topic: '),
                          member_keys=cid_topic.get(e['id'], []),
                          members=L(rows[c] for c in cid_topic.get(e['id'], [])[:members] if c in rows))
                 for e in ents)
    return AttrDict(clusters=clusters, method='cached', note=f'{len(clusters)} cached topics from graph')

@patch
def map(self:Vault, min_count:int=2, force:bool=False, **kw) -> AttrDict:
    'Cluster the corpus into labelled topics — the shape of what you have collected. Fast after `connect()` has run: reads the persisted topic nodes rather than re-clustering.'
    if not force:
        cached = _map_from_graph(self.db, self.name, self.store)
        if cached is not None: return cached
    return self.store.clusters(min_count=min_count, dtype=DTYPE, columns=['content', 'doc_id'], **kw)

def _sql_in(col, xs, batch:int=2000):
    "`col IN (...)` clauses over `xs`, split so no one statement grows unbounded."
    xs = list(xs)
    for i in range(0, len(xs), batch):
        yield f"{col} IN ({','.join(repr(x) for x in xs[i:i+batch])})"

@patch
def topic_tree(self:Vault,
               limit:int=20,      # topics returned, largest first
               docs:int=8,        # documents listed under each topic
               min_chunks:int=2,  # skip a topic carried by fewer chunks than this
) -> L:
    '''Topics, and which documents each one runs through. The shape of the corpus, two levels deep.

    `map()` says what the subjects are; this says where each one lives, which is the half that tells
    you whether a subject is one source talking to itself or a thread running through six. Reads the
    topic nodes `connect()` persisted, so it costs three queries and no clustering.'''
    try: g = self.db.get_graph(self.name)
    except Exception: return L()
    try: ents = L(g.entities(where="kind='topic'", order_by='freq desc'))
    except Exception: return L()
    if not ents: return L()
    by_topic = {}
    for w in _sql_in('entity_id', [e['id'] for e in ents]):
        for m in g.mentions(select='chunk_id, entity_id', where=w):
            by_topic.setdefault(m['entity_id'], []).append(m['chunk_id'])
    cid_doc = {}
    for w in _sql_in('id', {c for cs in by_topic.values() for c in cs}):
        for r in self.store(select='id, doc_id', where=w): cid_doc[r['id']] = r['doc_id']
    titles = {r['id']: r['title'] for r in self.t.docs(select='id, title')}
    out = L()
    for e in ents:
        cs = by_topic.get(e['id'], [])
        if len(cs) < min_chunks: continue
        n = Counter(d for c in cs if (d := cid_doc.get(c)))
        out.append(AttrDict(label=(e['content'] or '').removeprefix('topic: '), topic_id=e['id'],
                            size=e['freq'], chunks=len(cs), docs=len(n),
                            sources=L(AttrDict(doc_id=d, title=titles.get(d) or d, chunks=k)
                                      for d, k in n.most_common(docs))))
        if len(out) >= limit: break
    return out

def fmt_topics(tree, width:int=44) -> str:
    "A `topic_tree` as an indented listing. Plain ASCII, so it survives a terminal, a notebook and a prompt."
    if not tree: return 'no topics — run connect() first'
    lines = []
    for t in tree:
        lines.append(f"{t['label'][:width]:<{width}} ({t['chunks']} chunks, {t['docs']} docs)")
        srcs = t['sources']
        for i, s in enumerate(srcs):
            stem = '`- ' if i == len(srcs)-1 else '|- '
            lines.append(f"  {stem}{str(s['title'])[:width-4]:<{width-4}} {s['chunks']:>4}")
    return '\n'.join(lines)

@patch
def show_topics(self:Vault, limit:int=20, docs:int=8, **kw):
    "Print `topic_tree` as an indented listing."
    print(fmt_topics(self.topic_tree(limit=limit, docs=docs, **kw)))

@patch
def sources(self:Vault, kind:str=None) -> L:
    'Every document in the vault with its provenance, newest first.'
    rows = self.t.docs(where=_kw(kind) if kinds(kind) else None, order_by='added_at desc')
    return L(rows).map(lambda d: dict(d, meta=json.loads(d['meta'] or '{}')))

@patch
def forget(self:Vault, doc_id:str):
    'Remove a document, its sections and its chunks, and rebuild the ANN index.'
    self.db.delete_doc(doc_id, store=self.name)

@patch
def stats(self:Vault) -> dict:
    'Row counts across the vault, by kind.'
    p, t = self.t.prefix, self.db.t
    ents = (first(self.db.q(f"SELECT COUNT(*) n FROM {p}entities WHERE kind!='topic'")) or {}).get('n', 0) if f'{p}entities' in t else 0
    return dict(docs=self.t.docs.count, nodes=self.t.nodes.count, chunks=self.store.count, encoder=self.enc.method,
                entities=ents, path=self.path,
                by_kind={r['kind']: r['n'] for r in self.db.q(f'select kind, count(*) as n from {p}docs group by kind order by n desc')})

## Try it

`offline=True` skips the model download and uses litesearch's `hash_embed`, which is what you want
in CI and for a quick look: retrieval is lexical, and `stats()['encoder']` says so.

In [ ]:
v = Vault(':memory:')
v.note('Late chunking beats naive chunking because context survives the split.', tags=['retrieval'])
v.add('# Attention\n\nScaled dot-product attention weights values by query-key similarity.\n\n'
      '## Multi-head\n\nHeads attend to different subspaces in parallel.', 'Attention', kind='note')
v.stats()

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'docs': 2,
 'nodes': 5,
 'chunks': 3,
 'encoder': 'model2vec',
 'entities': 0,
 'path': ':memory:',
 'by_kind': {'note': 2}}

In [ ]:
test_eq(v.stats()['docs'], 2)
test_eq(v.stats()['encoder'], 'model2vec')
assert v.search('chunking')[0]['snippet']
# a hit is a handle plus enough text to recognise it, not the chunk itself — the scoring
# internals the two legs disagree about (`rank`, `_dist`, `_rrf_score`, `rowid`) stay inside
_h = v.search('chunking')[0]
test_eq(sorted(_h), ['breadcrumb', 'doc_id', 'node_id', 'page', 'score', 'snippet'])
assert len(v.search('chunking', chars=20)[0]['snippet']) <= 20
assert v.read(_h['node_id'])['text']          # the handle opens what the snippet came from
test_eq([d['kind'] for d in v.sources()], ['note', 'note'])
test_eq(len(v.sources(kind='web')), 0)
test_eq(len(v.search('chunking', kind='web')), 0)

In [ ]:
d = v.doc('Attention')                          # by title substring
test_eq(v.doc(d['id'])['title'], 'Attention')   # by doc_id
test_eq(v.doc(d['source'])['title'], 'Attention')  # by source
test_eq(v.doc('nothing in here'), None)

whole = v.document('Attention')
test_eq((whole.origin, whole.nodes), ('vault', 3))   # the root, the heading, the subheading
# every section, in document order, with the headings that say what each one is
assert whole.text.index('Scaled dot-product') < whole.text.index('## Multi-head') < whole.text.index('subspaces')
test_eq(v.document('Attention', headings=False).text.find('## Multi-head'), -1)
test_eq(v.document('Attention', max_chars=20).text, whole.text[:20])
test_eq(v.document('Attention', max_chars=20).truncated, True)
# a `Pages n–m:` node title is build_tree's placeholder, not a heading the document wrote
v.add('Prose with no headings at all, long enough to chunk and to be stored in the vault.', 'plain')
assert '# Pages' not in v.document('plain').text and 'Prose with no' in v.document('plain').text

test_eq(v.set_meta(d['id'], doctype='paper')['doctype'], 'paper')
test_eq(v.doc(d['id'])['meta']['doctype'], 'paper')                 # survives the round trip
test_eq(v.set_meta(d['id'], reviewed=True)['doctype'], 'paper')     # merged, not replaced
test_fail(lambda: v.document('no such document'), contains='no document in the vault')

In [ ]:
#| hide
# resolving an encoder name is pure, so this costs no download: the alias, the kind, and the id
test_eq(enc_spec('science'), ('minishlab/potion-science-32M', 'static'))
test_eq(enc_spec('gemma')[1], 'onnx')
test_eq(enc_spec('gemma')[0]['model'], 'onnx-community/embeddinggemma-300m-ONNX')
test_eq(enc_spec(None)[0], DFLT_ENC)
test_eq(enc_spec('some-org/some-model'), ('some-org/some-model', 'static'))   # an id passes through

class _E:            # an embedder you built yourself: litesearch's doc_encoder takes anything with .encode
    def encode(self, xs): return np.zeros((len(xs), 8), dtype=np.float16)
test_eq(enc_spec(_E())[1], 'ready')
test_eq((mk_encoder(_E()).dims, mk_encoder(_E()).method), (8, 'model2vec'))
# litesearch's model dicts are AttrDicts too, so `Vault(encoder=embedding_gemma)` must not be read
# as an already-built encoder — that would hand `Index` a dict where an embedder belongs
assert not _is_enc(embedding_gemma) and _is_enc(mk_encoder(_E()))
test_eq(mk_encoder('no/such-model-at-all', offline=True).method, 'hash')

# the vault's default and a bare litesearch Index have to agree, or one file holds two vector
# spaces under one name and nothing says so
test_eq(DFLT_ENC, 'minishlab/potion-multilingual-128M')

In [ ]:
#| hide
#|eval: false
test_eq(enc_spec('onnx-community/embeddinggemma-300m-ONNX')[1], 'onnx')
for _alias, _m in ENCODERS.items():
    if isinstance(_m, dict): test_eq(enc_spec(_m['model'])[1], 'onnx')   # every ONNX alias, not just gemma
test_eq(enc_spec('some-org/some-model'), ('some-org/some-model', 'static'))

for _a, _m in ENCODERS.items():
    if isinstance(_m, str): assert '/' in _m and not Path(_m).exists(), _a
# SHELVES is a tuple of names now, not an encoder map: one encoder writes every shelf
assert isinstance(SHELVES, tuple) and 'sanskrit' in SHELVES and 'papers' in SHELVES
assert all(isinstance(s, str) for s in SHELVES)

In [ ]:
#| hide
# a model that will not load must degrade to the hash encoder rather than leaving a broken vault
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    e = mk_encoder('no/such-model-at-all')
test_eq(e.method, 'hash')
test_eq(e.model.encode(['probe']).shape, (1, e.dims))
test_eq(e.model.encode(['probe']).dtype, np.float16)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _bad = Vault(':memory:', encoder='no/such-model-at-all')
test_eq(_bad.enc.method, 'hash')                     # and the vault opens
test_eq(_bad.emb(['probe']).shape, (1, e.dims))      # embedding through Index still works

/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_49612/2672644707.py:20: UserWarning: could not load no/such-model-at-all (LocalEntryNotFoundError: Got: ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local i); falling back to hash_embed — retrieval will be lexical, not semantic
  warnings.warn(f'could not load {nm} ({type(e).__name__}: {str(e)[:100]}); falling back to hash_embed — retrieval will be lexical, not semantic')


In [ ]:
#| hide
# a shelf is a partition of one file: its own store and tree, one shared connection
sh = v.shelf('papers')
sh.add('Contextual chunk embeddings keep the document around the chunk.', 'a paper')
test_eq((sh.name, sh.db is v.db), ('papers', True))     # shared, or ':memory:' would be a second db
test_eq([s['store'] for s in v.shelves()], ['store', 'papers'])
test_eq(v.doc('a paper'), None)                         # a partition, not a second index over the same docs
test_eq(sh.doc('a paper')['title'], 'a paper')
test_eq(v.shelf('papers').enc.method, 'model2vec')      # reopened with the encoder that wrote it
test_eq(v.shelf('sanskrit').name, 'sanskrit')
# every shelf is one vector space wide, which is what lets a shelf be a plain Index
test_eq({s['dims'] for s in v.shelves()}, {v.enc.dims})
test_eq((v.route('arxiv').name, v.route('web').name), ('papers', 'store'))

# reopening a store with a *different* encoder is silent and total, so it has to be said out loud
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    v.shelf('papers', encoder=_E())
    assert any('meaningless' in str(x.message) for x in w), [str(x.message) for x in w]

# ...and what makes a library of shelves usable is reading across it
e = v.elsewhere('chunk embeddings')
test_eq(e.attrgot('store'), ['papers'])
assert e[0].breadcrumb.startswith('papers › ') and e[0].text
assert v.read(e[0].node_id, store='papers')['text']   # a citation into a shelf has to open
test_eq(v.read(e[0].node_id), {})                     # and the wrong store finds nothing
test_eq(v.elsewhere('chunk embeddings', shelves=['nowhere']), [])
# the sanskrit shelf above is registered but empty, and an empty shelf is skipped rather than opened
assert 'sanskrit' in v.shelves().attrgot('store')
test_eq(sh.elsewhere('late chunking', limit=1).attrgot('store'), ['store'])   # reads both ways
test_eq(len(sh.elsewhere('late chunking', limit=2)), 2)                      # limit is per shelf

c = v.context('chunk embeddings', sections=2, related=0, code=0)
test_eq(c.shelves, 1)
assert any(r.get('store') == 'papers' for r in c.results), c.results
test_eq(v.context('chunk embeddings', related=0, code=0, shelves=0).shelves, 0)

# a shelf's encoder cannot be migrated in place, so dropping is the supported way to change it
_dropped = v.drop_shelf('papers')
assert 'papers' in _dropped['dropped'] and 'papers_docs' in _dropped['dropped'], _dropped
test_eq([s['store'] for s in v.shelves()], ['store', 'sanskrit'])   # gone from the registry too
test_eq(v.elsewhere('chunk embeddings'), [])                        # and from every read across
test_fail(lambda: v.drop_shelf('store'), contains='refusing to drop')
sh = v.shelf('papers')                                              # rebuilt empty, and reusable
sh.add('Contextual chunk embeddings keep the document around the chunk.', 'a paper')
test_eq(sh.doc('a paper')['title'], 'a paper')

In [ ]:
#| hide
from tempfile import mkdtemp
td = Path(mkdtemp()).resolve()          # litesearch reports the resolved path; /var is a symlink
vf = Vault(str(td/'v.db'), offline=True)
test_eq(vf.assets(), td/'assets')
test_eq(vf.assets('paper'), td/'assets'/'paper')

p = td/'fusion_notes.md'
p.write_text('# Fusion\n\nRanks are fused because the legs share no vector space.\n')
r = vf.add_file(p)
test_eq(r['kind'], 'md')                              # litesearch reads it off the extension
test_eq(r['title'], 'fusion notes')                   # and prettifies the filename
assert vf.search('rank fusion')

r = vf.add_file(p, title='Fusion, filed', kind='note')
test_eq(r['kind'], 'note')                            # an explicit kind wins, at ingest not after
test_eq(first(vf.t.docs(where=f'id={r["doc_id"]!r}'))['kind'], 'note')   # and it is what landed

# a shelf shares the vault's database, so it shares the vault's asset directory
test_eq(vf.shelf('papers', offline=True).assets(), vf.assets())
# an in-memory vault has no directory of its own; assets must not fall back to the cwd
assert not str(Vault(':memory:', offline=True).assets()).startswith(str(Path.cwd()))

# `embed_batch` decides *when* chunks are written, not which ones: batched across documents and
# written one document at a time have to land the same corpus, or the fast path is a different index
dd = td/'batched'; dd.mkdir()
for _i in range(6): (dd/f'n{_i}.md').write_text(f'# Note {_i}\n\nReciprocal rank fusion merges two ranked lists.\n')
_fs = sorted(dd.glob('*.md'))
vb = Vault(str(td/'b.db'), offline=True); vb.add_files(_fs, embed_batch=2)
vs = Vault(str(td/'s.db'), offline=True); vs.add_files(_fs, embed_batch=0, n_workers=0)
test_eq(vb.store.count, vs.store.count)
test_eq(vb.stats()['docs'], 6)
assert vb.search('rank fusion')        # and the ANN index was rebuilt, once, after the last flush
test_eq(len(vb.add_files(_fs)), 6)   # content-addressed, so a second pass re-reports rather than duplicates
test_eq(vb.stats()['docs'], 6)

# the parse pool is the other branch, and `n_workers` is the only way to reach it from a corpus of
# markdown — `None` counts parse-heavy extensions and picks serial for these. Same corpus either way.
vp = Vault(str(td/'p.db'), offline=True); _op = vp.add_files(_fs, n_workers=4)
test_eq((vp.stats()['docs'], vp.store.count), (6, vs.store.count))
test_eq(sum(o is None for o in _op), 0)      # every file came back, parsed in a worker or here
assert vp.search('rank fusion')
test_eq(vb.add_files([]), L())               # and no files does no work, not an index rebuild

# `batch` streams the corpus and keeps the co-occurrence windows on disk rather than in memory: a
# different place to keep the same numbers, so the graph it lands has to be the same graph. Compare
# the tables, not the return: the batched count sums its flushes, and two chunks with identical text
# share a derived id, so a duplicate is counted once unbatched and once per flush batched.
def _graph_of(batch):
    _vg = Vault(str(td/f'g{batch}.db'), offline=True)
    _vg.add_files(_fs); _vg.connect(batch=batch)
    _g = _vg.db.get_graph(_vg.name)
    return dict(entities=_g.entities.count, mentions=_g.mentions.count, edges=_g.edges.count,
                n=sum(m['n'] for m in _g.mentions(select='n')))
test_eq(_graph_of(0), _graph_of(1))     # unbatched, and a flush per chunk
test_eq(_graph_of(0), _graph_of(4))


In [ ]:
test_eq(tidy_bc('Attention › Pages 1–1: Scaled dot-product › Multi-head'), 'Attention › Multi-head')
test_eq(tidy_bc(None), '')
# a window with a blank first line leaves the placeholder bare, and it is still a placeholder
test_eq(tidy_bc('Doc › Pages 1–1: › Body'), 'Doc › Body')

In [ ]:
r = v.connect()
assert r['entities'] > 0 and r['resolved']['resolvable'] == r['entities']
test_eq(v.stats()['entities'], r['entities'])

In [ ]:
#| hide
# topic_tree is the other half of map(): map says what the subjects are, this says where each lives
_tt = v.topic_tree(limit=3, docs=2)
assert _tt, 'connect() ran, so there are topic nodes to read'
test_eq(sorted(_tt[0]), ['chunks', 'docs', 'label', 'size', 'sources', 'topic_id'])
assert all(t['chunks'] >= 2 for t in _tt)                    # min_chunks is honoured
assert all(len(t['sources']) <= 2 for t in _tt)              # and so is the per-topic doc cap
assert all(s['title'] for t in _tt for s in t['sources'])    # every source resolves to a real title
test_eq(len(v.topic_tree(limit=1)), 1)
assert 'topic: ' not in _tt[0]['label']                      # the storage prefix is not the label

# the renderer is plain ascii, and says so rather than raising when there is nothing to show
_s = fmt_topics(_tt)
assert _tt[0]['label'][:20] in _s and _s.isascii(), _s
test_eq(fmt_topics([]), 'no topics — run connect() first')
# a vault with no graph at all must answer, not explode
test_eq(Vault(':memory:', offline=True).topic_tree(), [])

In [ ]:
#| hide
test_eq(v.enc.dims, 256)
test_eq(len(v.qemb('chunking')), v.enc.dims*2)      # float16 bytes, matching what the store holds
test_eq(v.emb(['a', 'b']).shape, (2, v.enc.dims))
test_eq(v.emb(['a']).dtype, DTYPE)
with warnings.catch_warnings():
    warnings.simplefilter('error')          # litesearch warns on a dtype mismatch; it must not fire
    assert v.context('why does late chunking help', sections=2, related=2,
                     code=0, shelves=0).results   # this cell is about *this* store's width

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()